# Lab 12: Ingestion at scale

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Project-Lantern-Case-Study-Lab2-Team-2 | LLM mode: replay


The classroom demonstrations always use the fictional sample CIK `0000000001` and local JSON fixtures. They never contact the SEC.

`ONLINE = False` is the default: the final Apple exercise replays a snapshot of the four filings already obtained in your live run. This snapshot contains filtered results, not the original SEC response. Set `ONLINE = True` only to refresh the Apple exercise from SEC submissions (including overlapping archived batches). No global requests functions are replaced, so changing modes does not require restarting the kernel.

In [2]:
ONLINE = False  # Only controls the final Apple exercise.
import requests

SAMPLE_CIK = "0000000001"
H = {"User-Agent": "NEU-DAMG7245-2 nandavarapu.s@northeastern.edu"}

def sec_json(url):
    # A Session avoids the old notebook's requests.get monkeypatch even in an
    # existing kernel. HTTP errors are reported before JSON decoding.
    with requests.Session() as session:
        response = session.get(url, headers=H, timeout=30)
        response.raise_for_status()
        try:
            return response.json()
        except ValueError as exc:
            content_type = response.headers.get("Content-Type", "unknown")
            raise RuntimeError(
                f"SEC returned non-JSON data for {url} "
                f"(HTTP {response.status_code}, Content-Type: {content_type})."
            ) from exc

def edgar_data(cik, resource, *, online=False):
    cik = str(cik).zfill(10)
    if resource not in {"submissions", "companyfacts"}:
        raise ValueError(f"Unsupported resource: {resource}")
    if cik == SAMPLE_CIK:
        # The sample CIK is never sent to SEC, even when ONLINE is True.
        return json.loads(Path(f"data/edgar_fixture/{resource}_SAMPLE.json").read_text())
    if not online:
        raise ValueError("Real-company data requires online=True; use the saved Apple exercise snapshot offline.")
    if resource == "submissions":
        url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    else:
        url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    return sec_json(url)

In [3]:
# The submissions API: find exactly the filings you need.
def submissions(cik: str, *, online=False) -> dict:
    return edgar_data(cik, "submissions", online=online)

def earnings_8ks(cik: str, n: int = 4, *, online=False) -> list[dict]:
    r = submissions(cik, online=online)["filings"]["recent"]
    rows = zip(r["form"], r["items"], r["filingDate"],
               r["accessionNumber"], r["primaryDocument"])
    out = [dict(form=f, items=i, date=d, accn=a, doc=p)
           for f, i, d, a, p in rows
           if f == "8-K" and "2.02" in {x.strip() for x in i.split(",")}]
    return out[:n]

The next two demonstrations use **fictional sample data**, not Apple. The real-company exercise is at the end.

In [4]:
earnings_8ks(SAMPLE_CIK)

[{'form': '8-K',
  'items': '2.02,9.01',
  'date': '2026-08-05',
  'accn': '0000000001-26-000041',
  'doc': 'smpa-8k_20260805.htm'},
 {'form': '8-K',
  'items': '2.02,9.01',
  'date': '2026-05-06',
  'accn': '0000000001-26-000022',
  'doc': 'smpa-8k_20260506.htm'}]

In [5]:
# From the slide: companyfacts at scale: facts, not documents
import pandas as pd
KEEP = ("start", "end", "val", "fy", "fp", "form",
        "accn", "frame")

def facts_frame(cik: str, *, online=False) -> pd.DataFrame:
    j = edgar_data(cik, "companyfacts", online=online)
    rows = []
    for tax, concepts in j["facts"].items():
        for concept, body in concepts.items():
            for unit, obs in body["units"].items():
                for o in obs:
                    r = {k: o.get(k) for k in KEEP}
                    r.update(cik=cik, taxonomy=tax,
                             concept=concept, unit=unit)
                    rows.append(r)
    return pd.DataFrame(rows)

In [6]:
facts_frame(SAMPLE_CIK)[["concept", "unit", "end", "val", "fp", "form"]]

,concept,unit,end,val,fp,form
0,EntityCommonStockSharesOutstanding,shares,2026-07-31,61000000,Q2,10-Q
1,Revenues,USD,2025-12-31,820000000,FY,10-K
2,Revenues,USD,2026-06-30,430000000,Q2,10-Q
3,NetIncomeLoss,USD,2025-12-31,71000000,FY,10-K


The data room arrives as CS1-style records (`data/dataroom/deal_a/export/records.jsonl`), already in `lantern/2.0`. Value-tiered routing decides which path each source takes:

In [7]:
print(llm.P["ingest"]["routing"])
rec = sd.records("deal_a")[0]
print({k: rec[k] for k in ("schema", "doc_id", "deal_id", "source_type", "tier", "page", "block_id", "citation")})

{'comps': 'xbrl+docling_items', 'dataroom': 'cs1_full+docling+ocr', 'web': 'html_text'}
{'schema': 'lantern/2.0', 'doc_id': 'DR-deal_a-001', 'deal_id': 'deal_a', 'source_type': 'dataroom', 'tier': 4, 'page': 1, 'block_id': 'p0001_b001', 'citation': {'kind': 'pdf_bbox', 'doc_id': 'DR-deal_a-001', 'page': 1, 'bbox': [72.0, 91.5, 528.8, 143.0]}}


In [8]:
# From the slide: The synthetic web: crawl from the sitemap, inside an allowlist
import hashlib, xml.etree.ElementTree as ET
from urllib.parse import urlparse
from bs4 import BeautifulSoup
SM = "{http://www.sitemaps.org/schemas/sitemap/0.9}"

def sitemap_urls(xml_text):
    return [l.text.strip()
            for l in ET.fromstring(xml_text).iter(f"{SM}loc")]

def allowed(url, hosts):            # refuse everything else
    u = urlparse(url)
    return u.scheme == "https" and u.hostname in hosts

def web_record(url, html, deal_id):
    soup = BeautifulSoup(html, "html.parser")
    for t in soup(["script", "style", "nav"]):
        t.decompose()
    text = " ".join(soup.get_text(" ").split())
    sha = hashlib.sha256(html.encode()).hexdigest()
    return {"schema": "lantern/2.0", "doc_id": f"WEB-{sha[:8]}",
            "deal_id": deal_id, "source_type": "synthetic_web",
            "tier": 5, "text": text, "source_path": url,
            "citation": {"kind": "web_span", "url": url},
            "lineage": {"parents": [], "source_sha256": sha},
            "ingest_flags": []}

In [9]:
# Course-host URLs map to local files; this crawl always runs offline.
from urllib import robotparser
HOSTS = {llm.P["web"]["host"]}
urls = sitemap_urls(Path("data/web_site/sitemap.xml").read_text())
rp = robotparser.RobotFileParser()
rp.parse(Path("data/web_site/robots.txt").read_text().splitlines())
web, refused = [], []
for u in urls:
    if not allowed(u, HOSTS) or not rp.can_fetch("LANTERN", u):
        refused.append(u)
        print("refused:", u)
        continue
    html = Path("data/web_site", urlparse(u).path.lstrip("/")).read_text()
    web.append(web_record(u, html, "deal_a"))
assert len(web) == 3, "The supplied sitemap should yield three allowed pages."
assert "https://not-allowed.example.com/tracker.html" in refused
assert all(allowed(r["source_path"], HOSTS) for r in web)
print(f"{len(web)} web records; {len(refused)} refused URL(s)")
for record in web:
    print(record["source_path"], "|", record["doc_id"], "| tier", record["tier"])

refused: https://not-allowed.example.com/tracker.html
3 web records; 1 refused URL(s)
https://sim.example.edu/wren/about.html | WEB-ed15cb2c | tier 5
https://sim.example.edu/wren/press-2026-06.html | WEB-04cf1eea | tier 5
https://sim.example.edu/wren/careers.html | WEB-b8a1eb3d | tier 5


**Exercise 12 — Apple's earnings 8-Ks.**

The original live run found **four** Item 2.02 filings in the fixed, inclusive window **2025-10-14 through 2026-10-14**. This is the original exercise window, not a rolling year relative to today's date.

Offline runs load `data/edgar_fixture/apple_earnings_20251014_20261014.json`, which preserves the saved live result and its provenance. A live refresh (`ONLINE=True`) reads recent submissions and any overlapping archives, checks the returned company CIK, and saves the newly filtered result. Fictional sample fixtures are never substituted for Apple.

In [10]:
import time
import pandas as pd
from datetime import datetime, timezone

START = "2025-10-14"
END = "2026-10-14"
APPLE_CIK = "0000320193"
APPLE_SNAPSHOT = Path("data/edgar_fixture/apple_earnings_20251014_20261014.json")

def earnings_8ks_window(cik, start, end):
    data = submissions(cik, online=True)
    if str(data.get("cik", "")).zfill(10) != cik:
        raise ValueError("SEC response CIK does not match the requested company.")
    batches = [data["filings"]["recent"]]
    for archive in data["filings"].get("files", []):
        if archive["filingFrom"] <= end and archive["filingTo"] >= start:
            time.sleep(0.2)
            batches.append(sec_json(
                f"https://data.sec.gov/submissions/{archive['name']}"
            ))
    matches = []
    for batch in batches:
        for form, items, date, accession, document in zip(
            batch["form"],
            batch["items"],
            batch["filingDate"],
            batch["accessionNumber"],
            batch["primaryDocument"],
        ):
            item_codes = {item.strip() for item in items.split(",")}
            if form == "8-K" and "2.02" in item_codes and start <= date <= end:
                matches.append({
                    "date": date,
                    "form": form,
                    "items": items,
                    "accession": accession,
                    "document": document,
                })

    return (
        pd.DataFrame(matches, columns=[
            "date", "form", "items", "accession", "document"
        ])
        .drop_duplicates("accession")
        .sort_values("date")
        .reset_index(drop=True)
    )


if ONLINE:
    apple_filings = earnings_8ks_window(APPLE_CIK, START, END)
    snapshot = {
        "company": "Apple Inc.", "cik": APPLE_CIK,
        "start": START, "end": END,
        "source_url": f"https://data.sec.gov/submissions/CIK{APPLE_CIK}.json",
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "provenance": "Filtered live SEC submissions, including overlapping archives; not a raw response.",
        "filings": apple_filings.to_dict(orient="records"),
    }
else:
    snapshot = json.loads(APPLE_SNAPSHOT.read_text())
    if (snapshot["cik"], snapshot["start"], snapshot["end"]) != (APPLE_CIK, START, END):
        raise ValueError("Saved Apple snapshot does not match the requested company/date window.")
    apple_filings = pd.DataFrame(snapshot["filings"], columns=[
        "date", "form", "items", "accession", "document"
    ])

assert apple_filings["accession"].is_unique
assert apple_filings["accession"].str.startswith(APPLE_CIK + "-").all()
assert apple_filings["date"].between(START, END).all()
assert (apple_filings["form"] == "8-K").all()
assert apple_filings["items"].map(lambda s: "2.02" in {x.strip() for x in s.split(",")}).all()
if ONLINE:
    APPLE_SNAPSHOT.write_text(json.dumps(snapshot, indent=2) + "\n")
print("Source:", "live SEC refresh" if ONLINE else "saved live-result snapshot (offline)")
display(apple_filings)
print(
    f"Apple filed {len(apple_filings)} Form 8-K reports containing "
    f"Item 2.02 between {START} and {END}, inclusive."
)

Source: saved live-result snapshot (offline)


,date,form,items,accession,document
0,2025-10-30,8-K,"2.02,9.01",0000320193-25-000077,aapl-20251030.htm
1,2026-01-29,8-K,"2.02,9.01",0000320193-26-000005,aapl-20260129.htm
2,2026-04-30,8-K,"2.02,9.01",0000320193-26-000011,aapl-20260430.htm
3,2026-07-30,8-K,"2.02,9.01",0000320193-26-000018,aapl-20260730.htm


Apple filed 4 Form 8-K reports containing Item 2.02 between 2025-10-14 and 2026-10-14, inclusive.
